# J3 — Analyse des Acteurs

**Responsable** : Louis  
**Corpus** : `Dataset/data.xlsx` — 35 396 tweets, 19 mars → 1er mai 2026  
**Règle** : tout passe par `load_corpus()`, jamais de lecture directe du fichier Excel.

---

## Objectif

Explorer la dimension **acteurs** du corpus sans recours au LLM.
Les colonnes disponibles permettent une taxonomie proxy statistique :

| Type proxy | Critères |
|---|---|
| `vérifié` | `X Verified = True` → média, institution, personnalité publique |
| `influenceur` | `X Followers ≥ 10 000`, non vérifié |
| `actif` | `X Followers ∈ [1 000, 10 000[`, non vérifié |
| `standard` | `X Followers ∈ [100, 1 000[`, `X Posts > 100` |
| `compte suspect` | `X Posts ≤ 100` — compte peu actif, potentiellement récent/bot |

Ces catégories servent à :
1. Comprendre **qui parle** dans la crise
2. Identifier **qui porte** l'amplification
3. Détecter des **signaux suspects** (coordination, bots)
4. Alimenter les **recommandations pour `analyste.py`**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'figure.figsize':   (13, 5),
    'font.size':        12,
})

from src.tools.corpus_loader import load_corpus

df = load_corpus('../Dataset/data.xlsx')
df['_day'] = df['Date'].dt.date

FIGURES_DIR = Path('../slides/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Corpus : {len(df):,} tweets | {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Colonnes acteur disponibles : Author, X Verified, X Followers, X Posts, X Author ID')

---
## Section 1 — Taxonomie proxy des acteurs

Classification statistique sur `X Verified`, `X Followers` et `X Posts`.  
Aucun LLM — chaque tweet reçoit un type d'acteur dérivé de ses métadonnées.

In [ ]:
def classify_actor(row) -> str:
    followers = row.get('X Followers') or 0
    verified  = bool(row.get('X Verified', False))
    posts     = row.get('X Posts') or 0

    if verified:
        return 'vérifié'
    if followers >= 10_000:
        return 'influenceur'
    if followers >= 1_000:
        return 'actif'
    if posts <= 100:
        return 'compte suspect'
    return 'standard'


df['acteur_proxy'] = df.apply(classify_actor, axis=1)

dist = df['acteur_proxy'].value_counts()
print('=== Distribution des types d\'acteurs (par tweet) ===')
for t, n in dist.items():
    print(f'  {t:<18} {n:>7,}  ({n/len(df):.1%})')

print()
print(f'Auteurs uniques      : {df["Author"].nunique():,}')
print(f'Comptes vérifiés     : {df["X Verified"].sum():,} tweets ({df["X Verified"].mean():.1%})')

In [ ]:
COLORS = {
    'vérifié':        '#3b82f6',
    'influenceur':    '#8b5cf6',
    'actif':          '#10b981',
    'standard':       '#94a3b8',
    'compte suspect': '#ef4444',
}

# Par tweet (amplification) vs par auteur unique (présence)
dist_tweet  = df['acteur_proxy'].value_counts()
dist_author = df.drop_duplicates('Author')['acteur_proxy'].value_counts()

order = list(COLORS.keys())
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, dist, title in [
    (axes[0], dist_tweet,  'Distribution par tweet (amplification)'),
    (axes[1], dist_author, 'Distribution par auteur unique (présence)'),
]:
    vals   = [dist.get(t, 0) for t in order]
    colors = [COLORS[t] for t in order]
    bars   = ax.barh(order, vals, color=colors, height=0.6)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Nombre')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé → slides/figures/fig_acteurs_distribution.png')

---
## Section 2 — Activité temporelle par type d'acteur

Qui parle, et quand ? L'objectif est de voir si certains types d'acteurs
**précèdent** ou **accompagnent** les pics de crise.

In [ ]:
daily_by_type = (
    df.groupby(['_day', 'acteur_proxy'])
    .size()
    .unstack(fill_value=0)
)

# Normalisation : part de chaque type par jour
daily_share = daily_by_type.div(daily_by_type.sum(axis=1), axis=0) * 100

dates = pd.to_datetime(daily_by_type.index)

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# Volume absolu
ax = axes[0]
bottom = np.zeros(len(dates))
for t in order:
    if t in daily_by_type.columns:
        ax.bar(dates, daily_by_type[t].values, bottom=bottom,
               label=t, color=COLORS[t], alpha=0.85)
        bottom += daily_by_type[t].values
ax.set_title('Volume journalier par type d\'acteur', fontweight='bold')
ax.set_ylabel('Tweets / jour')
ax.legend(loc='upper right', fontsize=9)

# Part relative
ax2 = axes[1]
bottom = np.zeros(len(dates))
for t in order:
    if t in daily_share.columns:
        ax2.bar(dates, daily_share[t].values, bottom=bottom,
                label=t, color=COLORS[t], alpha=0.85)
        bottom += daily_share[t].values
ax2.set_title('Part relative par type d\'acteur (% du volume journalier)', fontweight='bold')
ax2.set_ylabel('Part (%)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_temporel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Poids et engagement par type d'acteur

Qui porte réellement l'amplification ? Followers et engagement (Likes + Shares) par type.

In [ ]:
df['_interactions'] = df['Likes'].fillna(0) + df['Shares'].fillna(0)

# Médiane followers et engagement moyen par type
actor_stats = df.groupby('acteur_proxy').agg(
    n_tweets       =('acteur_proxy', 'count'),
    n_auteurs      =('Author', 'nunique'),
    followers_med  =('X Followers', 'median'),
    followers_max  =('X Followers', 'max'),
    engagement_moy =('_interactions', 'mean'),
    pct_retweet    =('Engagement Type', lambda x: (x == 'RETWEET').mean()),
    pct_negatif    =('Sentiment',       lambda x: (x == 'negative').mean()),
).round(2)

print('=== Stats par type d\'acteur ===')
print(actor_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('engagement_moy', 'Engagement moyen (Likes + Shares) / tweet', 'steelblue'),
    ('pct_retweet',    'Taux de retweet par type (%)',               'darkorange'),
    ('pct_negatif',    'Taux de sentiment négatif (%)',              'crimson'),
]

for ax, (col, title, color) in zip(axes, metrics):
    vals = actor_stats[col].reindex(order).fillna(0)
    if 'pct' in col:
        vals = vals * 100
    bars = ax.barh(order, vals.values, color=[COLORS[t] for t in order], height=0.6)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold', fontsize=11)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_width() + max(vals.values) * 0.02, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_engagement.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Top auteurs

Classement des auteurs les plus actifs et les plus viraux.

In [ ]:
# Top 15 auteurs les plus actifs (volume de posts)
top_active = (
    df.groupby('Author')
    .agg(
        n_tweets      =('Author', 'count'),
        acteur_proxy  =('acteur_proxy', 'first'),
        followers     =('X Followers', 'first'),
        verified      =('X Verified',  'first'),
        engagement    =('_interactions', 'sum'),
    )
    .sort_values('n_tweets', ascending=False)
    .head(15)
)

print('=== Top 15 auteurs les plus actifs ===')
print(top_active[['n_tweets', 'acteur_proxy', 'followers', 'verified', 'engagement']].to_string())

In [ ]:
# Top 15 auteurs les plus viraux (total engagement)
top_viral = (
    df[df['_interactions'] > 0]
    .groupby('Author')
    .agg(
        engagement    =('_interactions', 'sum'),
        n_tweets      =('Author', 'count'),
        acteur_proxy  =('acteur_proxy', 'first'),
        followers     =('X Followers', 'first'),
        verified      =('X Verified', 'first'),
    )
    .sort_values('engagement', ascending=False)
    .head(15)
)

print('=== Top 15 auteurs les plus viraux (Likes + Shares) ===')
print(top_viral[['engagement', 'n_tweets', 'acteur_proxy', 'followers', 'verified']].to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, df_top, col, title in [
    (axes[0], top_active, 'n_tweets',   'Top 15 auteurs — volume de tweets'),
    (axes[1], top_viral,  'engagement', 'Top 15 auteurs — total engagement'),
]:
    labels = [a[:20] for a in df_top.index]
    vals   = df_top[col].values
    colors = [COLORS.get(t, '#94a3b8') for t in df_top['acteur_proxy']]
    bars   = ax.barh(labels, vals, color=colors, height=0.7)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=t) for t, c in COLORS.items()]
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=9, title='Type acteur')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_top.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — Signaux suspects : comptes récents et coordination

Les `comptes suspects` (X Posts ≤ 100) sont le proxy principal pour détecter
des comptes récents, des bots, ou des acteurs créés pour la crise.  
On combine avec les signaux de coordination issus de `coord_sem.py`.

In [ ]:
suspects = df[df['acteur_proxy'] == 'compte suspect'].copy()

print('=== Comptes suspects (X Posts ≤ 100) ===')
print(f'  Tweets de comptes suspects  : {len(suspects):,} ({len(suspects)/len(df):.1%} du corpus)')
print(f'  Auteurs suspects uniques    : {suspects["Author"].nunique():,}')
print(f'  Taux de RT (suspects)       : {(suspects["Engagement Type"] == "RETWEET").mean():.1%}')
print(f'  Taux de RT (non-suspects)   : {(df[df["acteur_proxy"] != "compte suspect"]["Engagement Type"] == "RETWEET").mean():.1%}')
print()

# Part de suspects par jour
daily_total   = df.groupby('_day').size()
daily_suspect = suspects.groupby('_day').size().reindex(daily_total.index, fill_value=0)
daily_suspect_share = daily_suspect / daily_total * 100

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(pd.to_datetime(daily_suspect_share.index), daily_suspect_share.values,
       color='#ef4444', alpha=0.7, label='% comptes suspects')

# Volume journalier en transparence
ax2 = ax.twinx()
ax2.plot(pd.to_datetime(daily_total.index), daily_total.values,
         color='steelblue', linewidth=1.5, alpha=0.5, label='Volume total')
ax2.set_ylabel('Volume total tweets', color='steelblue', fontsize=10)

ax.set_title('Part quotidienne des comptes suspects vs volume total', fontweight='bold')
ax.set_ylabel('% comptes suspects', color='#ef4444', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_suspects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Rapid-fire par type d'acteur
rf = df.sort_values(['X Author ID', 'Date']).copy()
rf['_delta_s'] = rf.groupby('X Author ID')['Date'].diff().dt.total_seconds()

rapid_fire_authors = rf[rf['_delta_s'] <= 60]['X Author ID'].unique()
df['_rapid_fire'] = df['X Author ID'].isin(rapid_fire_authors)

rf_by_type = df.groupby('acteur_proxy')['_rapid_fire'].mean() * 100

print('=== Rapid-fire (posts enchaînés en ≤ 60s) par type d\'acteur ===')
print(f'  Comptes rapid-fire uniques : {len(rapid_fire_authors):,}')
print()
print('  % de tweets provenant de comptes rapid-fire, par type :')
for t, v in rf_by_type.sort_values(ascending=False).items():
    print(f'    {t:<18} {v:.1f}%')

---
## Section 6 — Matrice acteur × narratif (proxy)

Croisement entre type d'acteur et sentiment, comme approximation
du croisement acteur × narratif (la classification narrative complète est dans `J2_narratifs.ipynb`).

In [ ]:
# Matrice acteur × sentiment (proxy narratif)
matrix = (
    df.groupby(['acteur_proxy', 'Sentiment'])
    .size()
    .unstack(fill_value=0)
    .reindex(order)
)
matrix_pct = matrix.div(matrix.sum(axis=1), axis=0) * 100

print('=== Matrice acteur × sentiment (%) ===')
print(matrix_pct.round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(order))
width = 0.28
sent_colors = {'positive': '#22c55e', 'neutral': '#94a3b8', 'negative': '#ef4444'}

for i, sent in enumerate(['positive', 'neutral', 'negative']):
    if sent in matrix_pct.columns:
        ax.bar(x + i * width, matrix_pct[sent].values, width,
               label=sent, color=sent_colors[sent], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('% des tweets du type')
ax.set_title('Distribution du sentiment par type d\'acteur', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_acteurs_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Feedback `analyste.py` (sans modification du code)

Observations issues de l'exploration statistique, à destination de l'équipe.

### Ce que l'exploration révèle

**1. Le contexte follower manque au LLM**  
L'agent `analyste.py` classifie `acteur_type` à partir du texte et de `X Verified`,  
mais ne reçoit pas `X Followers`. Un compte à 500k followers classifié `militant`  
n'a pas le même poids qu'un anonyme à 80 followers — le LLM ne peut pas faire la distinction.  
**Piste** : ajouter `f:{row['X Followers']}` dans `_format_tweets()` dans `analyste.py`.

**2. `coordonné / bot suspect` absent de la taxonomie**  
Les 5 types valides (`media`, `militant`, `influenceur`, `anonyme`, `institution`) ne capturent
pas les comptes potentiellement automatisés ou créés pour la crise.  
L'exploration montre que les `comptes suspects` (X Posts ≤ 100) représentent une part
significative du corpus avec un taux de RT différent.  
**Piste** : ajouter `bot_suspect` comme 6ème catégorie dans `ACTEURS_VALIDES`, avec
un few-shot exemple sur les comptes récents/scripté.

**3. Taille du sample**  
L'agent tourne sur `tweets_sample` (un échantillon) et non le corpus complet.  
Les acteurs minoritaires (vérifiés, influenceurs) peuvent être sous-représentés si
l'échantillon n'est pas stratifié par type d'acteur.  
**Piste** : stratifier le sample par type proxy avant de le passer à l'analyste.

In [ ]:
# Vérification : représentativité du sample si pris aléatoirement
SAMPLE_SIZE = 200  # taille typique d'un sample analyste

sample_random     = df.sample(n=SAMPLE_SIZE, random_state=42)
sample_stratified = df.groupby('acteur_proxy', group_keys=False).apply(
    lambda g: g.sample(n=max(1, int(SAMPLE_SIZE * len(g) / len(df))), random_state=42)
)

print(f'=== Représentativité du sample (n={SAMPLE_SIZE}) ===')
print()
print('Type proxy      | Corpus   | Aléatoire | Stratifié')
print('-' * 55)
for t in order:
    p_corpus   = (df['acteur_proxy'] == t).mean() * 100
    p_random   = (sample_random['acteur_proxy'] == t).mean() * 100
    p_strat    = (sample_stratified['acteur_proxy'] == t).mean() * 100
    print(f'{t:<18} {p_corpus:>6.1f}%   {p_random:>8.1f}%   {p_strat:>8.1f}%')

---
## Synthèse — Chiffres clés et export

In [ ]:
json_path = '../slides/chiffres_cles.json'
with open(json_path) as f:
    chiffres = json.load(f)

chiffres['acteur_dominant']        = dist.index[0]
chiffres['pct_acteur_dominant']    = round(float(dist.iloc[0] / len(df)), 4)
chiffres['pct_comptes_suspects']   = round(float((df['acteur_proxy'] == 'compte suspect').mean()), 4)
chiffres['pct_verifies']           = round(float((df['acteur_proxy'] == 'vérifié').mean()), 4)
chiffres['pct_influenceurs']       = round(float((df['acteur_proxy'] == 'influenceur').mean()), 4)
chiffres['nb_auteurs_uniques']     = int(df['Author'].nunique())
chiffres['rapid_fire_accounts']    = int(len(rapid_fire_authors))

with open(json_path, 'w') as f:
    json.dump(chiffres, f, indent=2, ensure_ascii=False)

print('chiffres_cles.json mis à jour :')
actor_keys = [k for k in chiffres if 'acteur' in k or 'verif' in k or 'influenc' in k
              or 'suspect' in k or 'auteur' in k or 'rapid' in k]
for k in actor_keys:
    print(f'  {k:<35} {chiffres[k]}')

print()
print('=== Figures générées ===')
for f_name in ['fig_acteurs_distribution.png', 'fig_acteurs_temporel.png',
               'fig_acteurs_engagement.png', 'fig_acteurs_top.png',
               'fig_acteurs_suspects.png', 'fig_acteurs_sentiment.png']:
    print(f'  slides/figures/{f_name}')